## **Content Based:**

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Dummy data for demonstration
dummy_data = {
    'book_id': [101, 102, 103, 104, 105, 106],
    'title': [
        'Deep Learning with Python', 
        'Natural Language Processing in Action', 
        'Advanced Convolutional Neural Networks', 
        'Structural Dynamics for Engineers',
        'Introduction to Machine Learning',
        'Design of Concrete Structures'
    ],
    'author': ['François Chollet', 'Hobson Lane', 'Ian Goodfellow', 'John Smith', 'Andrew Ng', 'Arthur Nilson'],
    'genre': ['AI & Deep Learning', 'AI & NLP', 'AI & Vision', 'Civil Engineering', 'Machine Learning', 'Civil Engineering'],
    'abstract': [
        'Learn neural networks, deep learning, and computer vision using Keras and TensorFlow.',
        'A guide to NLP, text generation, and understanding human language with Python.',
        'Advanced computer vision techniques using CNNs for image classification and segmentation.',
        'Principles of structural dynamics, civil engineering, and concrete design.',
        'Basics of machine learning, algorithms, and data analysis.',
        'Comprehensive guide to designing reinforced concrete structures and civil engineering materials.'
    ]
}

df = pd.DataFrame(dummy_data)
display(df)

,book_id,title,author,genre,abstract
0,101,Deep Learning with Python,François Chollet,AI & Deep Learning,"Learn neural networks, deep learning, and comp..."
1,102,Natural Language Processing in Action,Hobson Lane,AI & NLP,"A guide to NLP, text generation, and understan..."
2,103,Advanced Convolutional Neural Networks,Ian Goodfellow,AI & Vision,Advanced computer vision techniques using CNNs...
3,104,Structural Dynamics for Engineers,John Smith,Civil Engineering,"Principles of structural dynamics, civil engin..."
4,105,Introduction to Machine Learning,Andrew Ng,Machine Learning,"Basics of machine learning, algorithms, and da..."
5,106,Design of Concrete Structures,Arthur Nilson,Civil Engineering,Comprehensive guide to designing reinforced co...


In [2]:
# TF-IDF Vectorizer
# stop_words='english' will remove common English words that may not be useful for distinguishing between documents
tfidf = TfidfVectorizer(stop_words='english')
df['soup'] = df['title'] + " " + df['author'] + " " + df['genre'] + " " + df['abstract']
# convert the abstracts into a TF-IDF matrix
tfidf_matrix = tfidf.fit_transform(df['soup'])

print(tfidf_matrix.shape)

# calcualte cosine similarity between the TF-IDF vectors
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

(6, 60)


In [3]:
def get_recommendations(book_id, cosine_sim=cosine_sim, df=df):
    if book_id not in df['book_id'].values:
        return {"error": "Book not found"}
        
    # the index of the book that matches the book_id
    idx = df.index[df['book_id'] == book_id].tolist()[0]
    
    # the similarity scores with all books
    sim_scores = list(enumerate(cosine_sim[idx]))
    
    # sort them by similarity score in descending order
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    sim_scores = sim_scores[1:4]
    
    # prepare the response (JSON Format)
    results = []
    for i in sim_scores:
        results.append({
            "recommended_book_id": int(df['book_id'].iloc[i[0]]),
            "title": df['title'].iloc[i[0]],
            "match_score": f"{round(i[1] * 100, 1)}%"
        })
        
    return results

# test the recommendation system with a sample book_id
test_book_id = 104
book_name = df[df['book_id'] == test_book_id]['title'].values[0]

print(f" books similar to {book_name} (ID: {test_book_id})")
print("=" * 60)
print(get_recommendations(test_book_id))

 books similar to Structural Dynamics for Engineers (ID: 104)
[{'recommended_book_id': 106, 'title': 'Design of Concrete Structures', 'match_score': '38.8%'}, {'recommended_book_id': 101, 'title': 'Deep Learning with Python', 'match_score': '0.0%'}, {'recommended_book_id': 102, 'title': 'Natural Language Processing in Action', 'match_score': '0.0%'}]


## **Collaborative Filtering:**

In [4]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, Flatten, Dot, Dense
from tensorflow.keras.models import Model
import warnings
warnings.filterwarnings('ignore')
# dummy ratings data (user_id, book_id, rating)
ratings_data = {
    'user_id': [1, 1, 1, 2, 2, 3, 3, 4, 4],
    'book_id': [101, 102, 103, 104, 105, 101, 104, 102, 105],
    'rating':  [5.0, 4.0, 5.0, 5.0, 4.0, 4.0, 5.0, 2.0, 3.0]
}
df_ratings = pd.DataFrame(ratings_data)

# Encoding user_ids and book_ids to integers (for embedding layers)
user_ids = df_ratings['user_id'].unique()
book_ids = df_ratings['book_id'].unique()

user2encoded = {x: i for i, x in enumerate(user_ids)}
book2encoded = {x: i for i, x in enumerate(book_ids)}

df_ratings['user_encoded'] = df_ratings['user_id'].map(user2encoded)
df_ratings['book_encoded'] = df_ratings['book_id'].map(book2encoded)

num_users = len(user2encoded)
num_books = len(book2encoded)

C:\Users\eslam\AppData\Roaming\Python\Python310\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [5]:
embedding_size = 50

user_input = Input(shape=(1,), name='user_input')
user_embedding = Embedding(num_users, embedding_size, name='user_embedding')(user_input)
user_vec = Flatten(name='flatten_users')(user_embedding)

book_input = Input(shape=(1,), name='book_input')
book_embedding = Embedding(num_books, embedding_size, name='book_embedding')(book_input)
book_vec = Flatten(name='flatten_books')(book_embedding)

prod = Dot(axes=1, name='dot_product')([user_vec, book_vec])

dense = Dense(128, activation='relu')(prod)
output = Dense(1, activation='linear', name='prediction')(dense) # Output is a single rating number

model = Model(inputs=[user_input, book_input], outputs=output)
model.compile(optimizer='adam', loss='mean_squared_error')

print("The model architecture:")
model.summary()

# this is where the model learns to predict ratings based on user and book interactions
history = model.fit(
    x=[df_ratings['user_encoded'].values, df_ratings['book_encoded'].values],
    y=df_ratings['rating'].values,
    batch_size=2,
    epochs=20,
    verbose=1
)


The model architecture:


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ user_input          │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ book_input          │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ user_embedding      │ (None, 1, 50)     │        200 │ user_input[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ book_embedding      │ (None, 1, 50)     │        250 │ book_input[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_users       │ (None, 50)        │          0 │ user_embedding[0… │
│ (Flatten)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_books       │ (None, 50)        │          0 │ book_embedding[0… │
│ (Flatten)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dot_product (Dot)   │ (None, 1)         │          0 │ flatten_users[0]… │
│                     │                   │            │ flatten_books[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 128)       │        256 │ dot_product[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ prediction (Dense)  │ (None, 1)         │        129 │ dense[0][0]       │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 835 (3.26 KB)

 Trainable params: 835 (3.26 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 17.7746 
Epoch 2/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 17.4913 
Epoch 3/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 17.1574 
Epoch 4/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 16.8031 
Epoch 5/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 16.4001 
Epoch 6/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 15.9831 
Epoch 7/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 15.5246 
Epoch 8/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 15.0189 
Epoch 9/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 14.4766 
Epoch 10/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 13.8562
Epoch 11/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 13.2156
Epoch 12/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 12.5139 
Epoch 13/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 11.7816 
Epoch 14/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 10.9484
Epoch 15/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 10.1412 
Epoch 16/20
5/5 ━━━━━━

In [6]:
import numpy as np
def get_collaborative_recommendations(user_id, model, df_ratings, num_recommendations=2):
    # 1. Check if the user exists in the system
    if user_id not in user2encoded:
        return []
    
    # 2. Get all unique books in the system
    all_books = df_ratings['book_id'].unique()
    
    # 3. Get the books that the user has already read and rated
    books_read_by_user = df_ratings[df_ratings['user_id'] == user_id]['book_id'].values
    
    # 4. Filter out the books the user has already read to find unread books
    books_not_read = [book for book in all_books if book not in books_read_by_user]
    
    if len(books_not_read) == 0:
        return []
    
    # 5. Encode the user and the unread books for the model
    user_encoded = user2encoded[user_id]
    books_not_read_encoded = [book2encoded[book] for book in books_not_read]
    
    # Create input arrays for the model prediction
    user_input_array = np.array([user_encoded] * len(books_not_read_encoded))
    book_input_array = np.array(books_not_read_encoded)
    
    # 6. Predict the ratings using the trained model
    predicted_ratings = model.predict([user_input_array, book_input_array]).flatten()
    
    # 7. Combine the predictions with book IDs and sort them in descending order
    book_rating_pairs = list(zip(books_not_read, predicted_ratings))
    book_rating_pairs = sorted(book_rating_pairs, key=lambda x: x[1], reverse=True)
    
    # 8. Prepare the final response payload for the Backend API
    top_books = book_rating_pairs[:num_recommendations]
    
    results = []
    for book_id, predicted_rating in top_books:
        clipped_rating = np.clip(float(predicted_rating), 1.0, 5.0)
        results.append({
            "recommended_book_id": int(book_id),
            "predicted_rating": round(float(predicted_rating), 1) # Expected rating out of 5
        })
        
    return results

test_user = 1
print(f"Recommended books for User {test_user} based on their taste:")
print("-" * 60)

recommendations = get_collaborative_recommendations(test_user, model, df_ratings)
print(recommendations)

Recommended books for User 1 based on their taste:
------------------------------------------------------------
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
[{'recommended_book_id': 104, 'predicted_rating': 2.1}, {'recommended_book_id': 105, 'predicted_rating': 1.9}]


## **Hybird System:**

In [7]:
import json

# The Upgraded Hybrid Logic Engine (with Demographic Cold Start)
def get_hybrid_recommendations(user_id, user_department, df_ratings, current_df, current_sim_matrix, cf_model, history_threshold=2):
    final_response = {
        "user_id": user_id,
        "department": user_department,
        "status": "success",
        "collaborative_picks": [],
        "content_based_picks": [],
        "message": ""
    }
    
    user_history = df_ratings[df_ratings['user_id'] == user_id]
    
    # 1. Cold Start Scenario (New User)
    if len(user_history) < history_threshold:
        final_response["message"] = f"Cold Start: User has no history. Recommending based on department ({user_department})."
        
        # Demographic Logic: Find books that match the user's department in the 'genre' column
        department_books = current_df[current_df['genre'].str.contains(user_department, case=False, na=False)]
        
        # If we find books for this department, pick the first one as the seed. 
        # If not, fallback to a general popular book (101).
        if not department_books.empty:
            default_book_id = int(department_books.iloc[0]['book_id'])
        else:
            default_book_id = 101 
            
        final_response["content_based_picks"] = get_recommendations(
            book_id=default_book_id, 
            df=current_df, 
            cosine_sim=current_sim_matrix
        )
        return final_response

    # 2. Active User Scenario (Has History)
    final_response["message"] = "Active User: Generating hybrid recommendations based on personal history."
    
    # Get Collaborative Recommendations
    final_response["collaborative_picks"] = get_collaborative_recommendations(
        user_id=user_id, 
        model=cf_model, 
        df_ratings=df_ratings, 
        num_recommendations=2
    )
    
    # Get Content-Based Recommendations based on the highest rated book
    highest_rated_book = user_history.sort_values(by='rating', ascending=False).iloc[0]['book_id']
    
    final_response["content_based_picks"] = get_recommendations(
        book_id=highest_rated_book, 
        df=current_df, 
        cosine_sim=current_sim_matrix
    )
    
    return final_response

# -----------------------------------------
# Testing the Demographic Logic
# -----------------------------------------

# Test 1: New User from Civil Engineering Department (User 98)
print("Testing NEW user from Civil Engineering:")
print("-" * 60)
civil_user_result = get_hybrid_recommendations(
    user_id=98, 
    user_department="Civil", # The Backend tells us this student is in Civil
    df_ratings=df_ratings, 
    current_df=df, 
    current_sim_matrix=cosine_sim, 
    cf_model=model
)
print(json.dumps(civil_user_result, indent=4))

print("\n" + "="*60 + "\n")

# Test 2: New User from AI/Computer Department (User 99)
print("Testing NEW user from AI Department:")
print("-" * 60)
ai_user_result = get_hybrid_recommendations(
    user_id=99, 
    user_department="AI", # The Backend tells us this student is in AI
    df_ratings=df_ratings, 
    current_df=df, 
    current_sim_matrix=cosine_sim, 
    cf_model=model
)
print(json.dumps(ai_user_result, indent=4))

Testing NEW user from Civil Engineering:
------------------------------------------------------------
{
    "user_id": 98,
    "department": "Civil",
    "status": "success",
    "collaborative_picks": [],
    "content_based_picks": [
        {
            "recommended_book_id": 106,
            "title": "Design of Concrete Structures",
            "match_score": "38.8%"
        },
        {
            "recommended_book_id": 101,
            "title": "Deep Learning with Python",
            "match_score": "0.0%"
        },
        {
            "recommended_book_id": 102,
            "title": "Natural Language Processing in Action",
            "match_score": "0.0%"
        }
    ],
    "message": "Cold Start: User has no history. Recommending based on department (Civil)."
}


Testing NEW user from AI Department:
------------------------------------------------------------
{
    "user_id": 99,
    "department": "AI",
    "status": "success",
    "collaborative_picks": [],
    "content

## **Save Models:**

In [8]:
import joblib
import os
import torch

os.makedirs('production_models', exist_ok=True)

model.save('production_models/collaborative_model.keras')

encoders = {'user2encoded': user2encoded, 'book2encoded': book2encoded}
joblib.dump(encoders, 'production_models/encoders.pkl')

joblib.dump(df, 'production_models/books_dataframe.pkl')
joblib.dump(cosine_sim, 'production_models/content_sim_matrix.pkl')
joblib.dump(df_ratings, 'production_models/df_ratings.pkl')

['production_models/df_ratings.pkl']